In [1]:
from google.colab import files
uploaded = files.upload()  # This will open a file dialog for you to upload the ZIP file


Saving Fake.csv to Fake.csv


In [2]:
import pandas as pd

# Load the extracted CSV file
df = pd.read_csv("/content/Fake.csv")

# Check the first few rows to verify
print(df.head())


                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  


In [17]:
from google.colab import files
uploaded = files.upload()  # Select true.csv from your system when prompted


Saving true.csv to true (1).csv


In [19]:
import pandas as pd

# Load the real news dataset
true_df = pd.read_csv("true.csv")

# Display first few rows
true_df.head()


,text
0,President signs new bill to support clean energy.
1,NASA announces the discovery of Earth-like exo...
2,Local schools see improvement in student perfo...
3,Government launches healthcare benefits for se...
4,Tech company unveils latest AI-powered gadget.


In [5]:
import pandas as pd

# Load the fake news dataset
fake_df = pd.read_csv("Fake.csv")
fake_df['label'] = 1  # Mark as fake news


In [20]:
# Add label columns
fake_df['label'] = 1  # Fake news
true_df['label'] = 0  # Real news

# Standardize the text column (use only 'text' and 'label')
fake_df = fake_df[['text', 'label']]
true_df = true_df[['text', 'label']]

# Reduce dataset size for faster training (10% from each)
fake_df_small = fake_df.sample(frac=0.1, random_state=42)
true_df_small = true_df.sample(frac=0.1, random_state=42)

# Combine both
df_small = pd.concat([fake_df_small, true_df_small], axis=0)

# Shuffle the combined dataset
df_small = df_small.sample(frac=1, random_state=42).reset_index(drop=True)

# Check the shape
print("Dataset size:", df_small.shape)
print(df_small.head())


Dataset size: (2349, 2)
                                                text  label
0  Says the guy who was a Jr. Senator and Communi...      1
1  A senior Donald Trump adviser is calling Hilla...      1
2  Who kicks homeless people out onto the streets...      1
3  Hillary was too busy to be bothered with makin...      1
4  Donald Trump set off a lot of alarm bells with...      1


In [21]:
true_df = pd.read_csv("true.csv")
true_df['label'] = 0  # Mark as real news


In [22]:
# Keep only relevant columns
fake_df = fake_df[['text', 'label']]
true_df = true_df[['text', 'label']]


In [23]:
# Combine both DataFrames
combined_df = pd.concat([fake_df, true_df], ignore_index=True)

# Shuffle the combined data
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

# Preview final dataset
combined_df.head()


,text,label
0,Donald Trump is so desperate to hear praise th...,1
1,"Whether you realize it or not, the Second Amen...",1
2,It s no secret that President Trump is very mu...,1
3,Thomas article can be read below:My dad calle...,1
4,The tax-exempt Muslim group CAIR has ties to t...,1


In [25]:
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Split your data into training and testing
from sklearn.model_selection import train_test_split

# Use the smaller dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_small['text'].tolist(), df_small['label'].tolist(),
    test_size=0.2, random_state=42
)

# Tokenize
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)



In [26]:
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification

# Create a custom Dataset class
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create dataset objects
train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# Load pre-trained BERT model for binary classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
from transformers import Trainer, TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    report_to='none'  # Disable WandB logging
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# Train the model
trainer.train()


Step,Training Loss
10,0.858800
20,0.533500
30,0.270900
40,0.076000
50,0.020900
60,0.004700
70,0.002700
80,0.000800
90,0.000600
100,0.000900


TrainOutput(global_step=470, training_loss=0.037769923370425645, metrics={'train_runtime': 373.3392, 'train_samples_per_second': 10.066, 'train_steps_per_second': 1.259, 'total_flos': 988771346042880.0, 'train_loss': 0.037769923370425645, 'epoch': 2.0})

In [28]:
# Evaluate the model
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)


Evaluation Results: {'eval_loss': 0.018548911437392235, 'eval_runtime': 13.6951, 'eval_samples_per_second': 34.319, 'eval_steps_per_second': 2.191, 'epoch': 2.0}


In [29]:
from transformers import pipeline

# Create a text classification pipeline using the trained model
fake_news_classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)

# Test with your own headline/text
sample_text = "NASA discovers water on the Moon's surface."
prediction = fake_news_classifier(sample_text)

print("Prediction:", prediction)


Device set to use cuda:0


Prediction: [{'label': 'LABEL_1', 'score': 0.999748170375824}]


In [ ]:
# Save model and tokenizer
trainer.model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")


In [34]:
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("saved_model")
tokenizer = BertTokenizer.from_pretrained("saved_model")

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)


Device set to use cuda:0
